In [11]:
import clickhouse_connect
client = clickhouse_connect.get_client(
  host='localhost', # IP server / URL ngrok
  port=8123,
  username='mahasiswa',
  password='bigdata123',
  database='praktikum',
  secure=False
)

print("Koneksi berhasil:", client.server_version)

create_table_query = """
CREATE TABLE IF NOT EXISTS ecommerce_events(
    event_time      DateTime64(3, 'UTC'),
    event_type      Enum8(
                        'view' = 1,
                        'cart' = 2,
                        'remove_from_cart' = 3,
                        'purchase' = 4
                    ),
    product_id      UInt64,
    category_id     UInt64,
    category_code   String DEFAULT '',
    brand           String DEFAULT '',
    price           Float64,
    user_id         UInt64,
    user_session    Nullable(String)
)
ENGINE = MergeTree
PARTITION BY toYYYYMM(event_time)
ORDER BY (user_id, event_time, event_type)
SETTINGS index_granularity = 8192;
"""

client.command(create_table_query)

print("Tabel berhasil dibuat!")

Koneksi berhasil: 26.3.9.8
Tabel berhasil dibuat!


In [3]:
import pandas as pd

In [4]:
chunk_size = 100_000

existing = client.query("SELECT count() FROM ecommerce_events").result_rows[0][0]

if existing > 0:
    print(f"Data sudah ada ({existing} baris), skip insert.")
else:
    total_rows = 0

    for chunk in pd.read_csv(
        "E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv",
        chunksize=chunk_size
    ):
        chunk.columns = [
            'event_time', 'event_type', 'product_id', 'category_id', 'category_code',
            'brand', 'price', 'user_id', 'user_session'
        ]

        # Convert tipe data
        chunk['event_time'] = pd.to_datetime(chunk['event_time'], utc=True)
        chunk['product_id'] = chunk['product_id'].astype('uint64')
        chunk['category_id'] = chunk['category_id'].astype('uint64')
        chunk['price'] = chunk['price'].astype('float64')
        chunk['user_id'] = chunk['user_id'].astype('uint64')

        chunk['category_code'] = chunk['category_code'].fillna('')
        chunk['brand'] = chunk['brand'].fillna('')
        chunk['user_session'] = chunk['user_session'].where(
            chunk['user_session'].notna(), None)

        valid_types = ['view', 'cart', 'remove_from_cart', 'purchase']
        chunk = chunk[chunk['event_type'].isin(valid_types)]

        # INSERT per chunk
        client.insert_df('ecommerce_events', chunk)

        total_rows += len(chunk)
        print(f"Inserted {len(chunk)} rows (Total: {total_rows})")

    total = client.query("SELECT count() FROM ecommerce_events").result_rows[0][0]
    print(f"Insert selesai! Total data: {total} baris")

Inserted 100000 rows (Total: 100000)
Inserted 100000 rows (Total: 200000)
Inserted 100000 rows (Total: 300000)
Inserted 100000 rows (Total: 400000)
Inserted 100000 rows (Total: 500000)
Inserted 100000 rows (Total: 600000)
Inserted 100000 rows (Total: 700000)
Inserted 100000 rows (Total: 800000)
Inserted 100000 rows (Total: 900000)
Inserted 100000 rows (Total: 1000000)
Inserted 100000 rows (Total: 1100000)
Inserted 100000 rows (Total: 1200000)
Inserted 100000 rows (Total: 1300000)
Inserted 100000 rows (Total: 1400000)
Inserted 100000 rows (Total: 1500000)
Inserted 100000 rows (Total: 1600000)
Inserted 100000 rows (Total: 1700000)
Inserted 100000 rows (Total: 1800000)
Inserted 100000 rows (Total: 1900000)
Inserted 100000 rows (Total: 2000000)
Inserted 100000 rows (Total: 2100000)
Inserted 100000 rows (Total: 2200000)
Inserted 100000 rows (Total: 2300000)
Inserted 100000 rows (Total: 2400000)
Inserted 100000 rows (Total: 2500000)
Inserted 100000 rows (Total: 2600000)
Inserted 100000 rows 

In [5]:
result = client.query("SELECT COUNT(*) FROM praktikum.ecommerce_events")
print(result.result_rows)

[(67501979,)]


In [6]:
# import data kedua
chunk_size = 100_000

existing = client.query("SELECT count() FROM ecommerce_events").result_rows[0][0]

if existing > 0:
    print(f"Data sudah ada ({existing} baris), skip insert.")
else:
    total_rows = 0

    for chunk in pd.read_csv(
        "E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Nov.csv",
        chunksize=chunk_size
    ):
        chunk.columns = [
            'event_time', 'event_type', 'product_id', 'category_id', 'category_code',
            'brand', 'price', 'user_id', 'user_session'
        ]

        # Convert tipe data
        chunk['event_time'] = pd.to_datetime(chunk['event_time'], utc=True)
        chunk['product_id'] = chunk['product_id'].astype('uint64')
        chunk['category_id'] = chunk['category_id'].astype('uint64')
        chunk['price'] = chunk['price'].astype('float64')
        chunk['user_id'] = chunk['user_id'].astype('uint64')

        chunk['category_code'] = chunk['category_code'].fillna('')
        chunk['brand'] = chunk['brand'].fillna('')
        chunk['user_session'] = chunk['user_session'].where(
            chunk['user_session'].notna(), None)

        valid_types = ['view', 'cart', 'remove_from_cart', 'purchase']
        chunk = chunk[chunk['event_type'].isin(valid_types)]

        # INSERT per chunk
        client.insert_df('ecommerce_events', chunk)

        total_rows += len(chunk)
        print(f"Inserted {len(chunk)} rows (Total: {total_rows})")

    total = client.query("SELECT count() FROM ecommerce_events").result_rows[0][0]
    print(f"Insert selesai! Total data: {total} baris")

Data sudah ada (67501979 baris), skip insert.


In [7]:
# untuk insert data baru

def load_csv(path):
    chunk_size = 1_000_000
    total_rows = 0

    for chunk in pd.read_csv(path, chunksize=chunk_size):
        chunk.columns = [
            'event_time', 'event_type', 'product_id', 'category_id', 'category_code',
            'brand', 'price', 'user_id', 'user_session'
        ]

        chunk['event_time'] = pd.to_datetime(chunk['event_time'], utc=True)
        chunk['product_id'] = chunk['product_id'].astype('uint64')
        chunk['category_id'] = chunk['category_id'].astype('uint64')
        chunk['price'] = chunk['price'].astype('float64')
        chunk['user_id'] = chunk['user_id'].astype('uint64')

        chunk['category_code'] = chunk['category_code'].fillna('')
        chunk['brand'] = chunk['brand'].fillna('')
        chunk['user_session'] = chunk['user_session'].where(
            chunk['user_session'].notna(), None
        )

        valid_types = ['view', 'cart', 'remove_from_cart', 'purchase']
        chunk = chunk[chunk['event_type'].isin(valid_types)]

        client.insert_df('ecommerce_events', chunk)

        total_rows += len(chunk)
        print(f"{path} → inserted {total_rows}")

    print(f"Selesai: {path}")

In [8]:
load_csv("E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv")

E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 1000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 2000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 3000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 4000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 5000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 6000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 7000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 8000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 9000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 10000000
E:/Informatics Engineering Collage/Semester 6/Big Data/2019-Oct.csv → inserted 11000000
E:/Informatics Engineering Collage/Semest

In [14]:
result = client.query("SELECT COUNT(*) FROM praktikum.ecommerce_events")
print(result.result_rows)

[(109950743,)]


In [15]:
## EDA SQL

In [16]:
# Statistik
df_stats = client.query_df("""
SELECT
    count() AS total_rows,
    uniqExact(user_id) AS unique_users,
    uniqExact(product_id) AS unique_products,
    min(event_time) AS start_time,
    max(event_time) AS end_time
FROM ecommerce_events
""")

print(df_stats)

   total_rows  unique_users  unique_products start_time            end_time
0   109950743       5316649           206876 2019-10-01 2019-11-30 23:59:59


In [17]:
# distribusi event_type
df_event = client.query_df("""
SELECT
    event_type,
    count() AS total,
    round(count() * 100.0 / sum(count()) OVER (), 2) AS percentage
FROM ecommerce_events
GROUP BY event_type
ORDER BY total DESC
""")

print(df_event)

  event_type      total  percentage
0       view  104335509       94.89
1       cart    3955446        3.60
2   purchase    1659788        1.51


In [18]:
# statistik harga

df_price = client.query_df("""
SELECT
    min(price) AS min_price,
    max(price) AS max_price,
    avg(price) AS avg_price,
    quantile(0.25)(price) AS q1,
    quantile(0.5)(price) AS median,
    quantile(0.75)(price) AS q3
FROM ecommerce_events
WHERE price IS NOT NULL
""")

print(df_price)

   min_price  max_price   avg_price      q1  median      q3
0        0.0    2574.07  291.634802  67.635  165.51  360.11


In [19]:
# missing value

df_missing = client.query_df("""
SELECT
    count() AS total,
    countIf(category_code = '' OR isNull(category_code)) AS missing_category,
    countIf(brand = '' OR isNull(brand)) AS missing_brand,
    countIf(price IS NULL) AS missing_price,
    countIf(user_session IS NULL OR user_session = '') AS missing_session
FROM ecommerce_events
""")

print(df_missing)

       total  missing_category  missing_brand  missing_price  missing_session
0  109950743          35413780       15341158              0               12


In [20]:
# distribusi per-jam
df_hour = client.query_df("""
SELECT
    toHour(event_time) AS hour,
    count() AS total
FROM ecommerce_events
GROUP BY hour
ORDER BY hour
""")

print(df_hour)

    hour    total
0      0   755539
1      1  1402424
2      2  2745296
3      3  3949563
4      4  4982552
5      5  5569872
6      6  5823993
7      7  5920460
8      8  6079148
9      9  6005441
10    10  5924965
11    11  5776755
12    12  5726176
13    13  6263336
14    14  7013663
15    15  7431991
16    16  7565811
17    17  7149551
18    18  5391767
19    19  3716881
20    20  2175813
21    21  1263940
22    22   753640
23    23   562166


In [21]:
# tabel untuk cleaning

client.command('''
CREATE TABLE IF NOT EXISTS ecommerce_events_clean
(
    event_time DateTime64(3, 'UTC'),
    event_type Enum8(
        'view' = 1,
        'cart' = 2,
        'remove_from_cart' = 3,
        'purchase' = 4
    ),
    product_id UInt64,
    category_id UInt64,
    category_code String,
    brand String,
    price Float64,
    user_id UInt64,
    user_session String
)
ENGINE = MergeTree()
ORDER BY (user_id, event_time)
''')

In [22]:
# hitung median harga
median_result = client.query("""
SELECT quantile(0.5)(price) 
FROM ecommerce_events 
WHERE price IS NOT NULL
""")

median_price = median_result.result_rows[0][0]
print(f"Median price: {median_price}")

Median price: 164.695


In [23]:
# cleaning data per-partisi(bulan)

partitions = client.query("""
SELECT DISTINCT toYYYYMM(event_time) 
FROM ecommerce_events
ORDER BY 1
""").result_rows

for (p,) in partitions:
    print(f"Processing partition {p}")

    insert_query = f'''
    INSERT INTO ecommerce_events_clean
    SELECT
        event_time,
        event_type,
        product_id,
        category_id,

        if(category_code = '' OR isNull(category_code), 'unknown', category_code),
        if(brand = '' OR isNull(brand), 'unknown', brand),

        ifNull(price, {median_price}),

        user_id,

        if(user_session = '' OR isNull(user_session), 'no_session', user_session)

    FROM ecommerce_events
    WHERE toYYYYMM(event_time) = {p}
    '''

    client.command(insert_query)

    print(f"Selesai {p}")

Processing partition 201910
Selesai 201910
Processing partition 201911
Selesai 201911


In [24]:
# check data apakah sudah bersih
df_check = client.query_df("""
SELECT
    count() AS total,

    countIf(category_code = '' OR isNull(category_code)) AS bad_category,
    countIf(brand = '' OR isNull(brand)) AS bad_brand,
    countIf(price IS NULL) AS bad_price,
    countIf(user_session IS NULL OR user_session = '') AS bad_session

FROM ecommerce_events_clean
""")

print(df_check)

       total  bad_category  bad_brand  bad_price  bad_session
0  219901486             0          0          0            0


In [25]:
df_funnel = client.query_df("""
SELECT
    event_type,
    count() AS total
FROM ecommerce_events_clean
GROUP BY event_type
ORDER BY total DESC
""")

print(df_funnel)

  event_type      total
0       view  208671018
1       cart    7910892
2   purchase    3319576


In [26]:
df_revenue = client.query_df("""
SELECT
    sum(price) AS total_revenue,
    countIf(event_type = 'purchase') AS total_purchase
FROM ecommerce_events_clean
""")

print(df_revenue)

   total_revenue  total_purchase
0   6.413093e+10         3319576


In [27]:
df_hour = client.query_df("""
SELECT
    toHour(event_time) AS hour,
    count() AS total
FROM ecommerce_events_clean
GROUP BY hour
ORDER BY hour
""")

print(df_hour)

    hour     total
0      0   1511078
1      1   2804848
2      2   5490592
3      3   7899126
4      4   9965104
5      5  11139744
6      6  11647986
7      7  11840920
8      8  12158296
9      9  12010882
10    10  11849930
11    11  11553510
12    12  11452352
13    13  12526672
14    14  14027326
15    15  14863982
16    16  15131622
17    17  14299102
18    18  10783534
19    19   7433762
20    20   4351626
21    21   2527880
22    22   1507280
23    23   1124332


In [28]:
# # Query untuk menguji Feature Engineering (ekstraksi waktu dan hashing teks ke angka)
# query_test = """
# SELECT
#     toHour(event_time) AS event_hour,
#     toDayOfWeek(event_time) AS event_day,
    
#     -- Mengambil main category (misal: electronics) lalu di-hash menjadi angka ID
#     cityHash64(if(length(splitByChar('.', category_code)) > 1, splitByChar('.', category_code)[1], category_code)) AS category_id_hash,
    
#     -- Mengubah string brand menjadi angka ID
#     cityHash64(brand) AS brand_id_hash,
    
#     price,
#     event_type
# FROM ecommerce_events_clean
# LIMIT 10
# """

# # Mengeksekusi query dan memuatnya ke Pandas DataFrame
# print("Menarik 10 baris sampel fitur ML dari ClickHouse...")
# df_test = client.query_df(query_test)

# # Menampilkan hasil
# df_test

In [29]:
import numpy as np

In [30]:
print("Menyuruh ClickHouse merangkum 110 juta baris menjadi Profil Pengguna...")
print("Mohon tunggu, ini membutuhkan waktu beberapa menit...")

# Query SQL untuk merangkum riwayat setiap user
query_user_profile = """
SELECT
    user_id,
    countIf(event_type = 'view') AS view_count,
    countIf(event_type = 'cart') AS cart_count,
    countIf(event_type = 'purchase') AS purchase_count,
    sumIf(price, event_type = 'purchase') AS total_spend,
    uniqExact(category_code) AS distinct_categories,
    uniqExact(toYYYYMMDD(event_time)) AS active_days
FROM ecommerce_events_clean
GROUP BY user_id
"""
df_user = client.query_df(query_user_profile)

# Membersihkan nilai kosong (Jika user tidak pernah beli, total_spend = 0)
df_user['total_spend'] = df_user['total_spend'].fillna(0)

print(f"Selesai! Berhasil membentuk {len(df_user):,} profil pelanggan unik.")
# Menampilkan 5 sampel data pertama
df_user.head(30)

Menyuruh ClickHouse merangkum 110 juta baris menjadi Profil Pengguna...
Mohon tunggu, ini membutuhkan waktu beberapa menit...
Selesai! Berhasil membentuk 5,316,649 profil pelanggan unik.


,user_id,view_count,cart_count,purchase_count,total_spend,distinct_categories,active_days
0,512744610,4,0,0,0.00,2,2
1,540965351,18,0,0,0.00,1,4
2,556896890,6,0,0,0.00,1,2
3,549337691,4,0,0,0.00,1,1
4,524269480,6,0,0,0.00,3,1
5,547073191,70,0,0,0.00,2,3
6,567025803,2,0,0,0.00,1,1
7,571005635,30,10,4,257.40,2,1
8,575898479,36,0,0,0.00,3,2
9,544809108,22,0,0,0.00,3,3


In [31]:
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

In [32]:
#Klustering K-Means Model

In [33]:
print("Melakukan normalisasi skala data (Scaling)...")

# Kita memilih fitur perilaku
features = [
    'view_count', 
    'cart_count', 
    'purchase_count', 
    'total_spend', 
    'distinct_categories', 
    'active_days'
]

X_cluster = df_user[features]

# Menggunakan RobustScaler sangat untuk mengatasi outlier
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_cluster)

print("Data berhasil dinormalisasi dan siap dimasukkan ke mesin K-Means!")

Melakukan normalisasi skala data (Scaling)...
Data berhasil dinormalisasi dan siap dimasukkan ke mesin K-Means!


In [34]:
print("Memulai proses K-Means Clustering untuk membentuk 4 Persona Pelanggan...")

# n_init='auto' membuat algoritma lebih cepat di versi scikit-learn terbaru
kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')

df_user['cluster'] = kmeans.fit_predict(X_scaled)

print("Klustering selesai! Setiap pelanggan kini telah memiliki label segmennya masing-masing.")

Memulai proses K-Means Clustering untuk membentuk 4 Persona Pelanggan...
Klustering selesai! Setiap pelanggan kini telah memiliki label segmennya masing-masing.


In [35]:
print("Membongkar Karakteristik Rata-rata dari Setiap Klaster...")

# Menghitung rata-rata fitur per klaster
kmeans_profile = df_user.groupby('cluster')[features].mean().round(2)

# Menghitung populasi (jumlah orang) di masing-masing klaster
kmeans_profile['jumlah_orang'] = df_user.groupby('cluster')['user_id'].count()

# Mengurutkan dari klaster yang paling ramai populasinya ke yang paling sepi
kmeans_profile = kmeans_profile.sort_values(by='jumlah_orang', ascending=False)

# Menampilkan hasil akhir
kmeans_profile

Membongkar Karakteristik Rata-rata dari Setiap Klaster...


,view_count,cart_count,purchase_count,total_spend,distinct_categories,active_days,jumlah_orang
cluster,,,,,,,
0,38.00,1.27,0.45,99.79,2.31,2.78,5271330
2,171.57,22.82,17.45,8275.52,5.01,9.51,42620
1,370.07,75.01,68.95,42101.14,6.90,19.42,2587
3,944.14,237.93,250.70,202177.50,8.03,32.54,112


In [36]:
print("=== 1. PERSIAPAN SAMPLING UNTUK EVALUASI ===")
# Mengambil 100.000 data acak secara konsisten untuk menyelamatkan RAM
np.random.seed(42)
sample_indices = np.random.choice(X_scaled.shape[0], size=100000, replace=False)

X_sample = X_scaled[sample_indices]
y_kmeans_sample = df_user['cluster'].iloc[sample_indices].values
y_gmm_sample = df_user['cluster_gmm'].iloc[sample_indices].values

print("Sampling 100.000 data berhasil! Siap untuk dievaluasi.")

=== 1. PERSIAPAN SAMPLING UNTUK EVALUASI ===


KeyError: 'cluster_gmm'

In [ ]:
print("=== 2. EVALUASI K-MEANS ===")

# Menghitung skor untuk K=4 (Model Final)
db_kmeans = davies_bouldin_score(X_sample, y_kmeans_sample)
sil_kmeans = silhouette_score(X_sample, y_kmeans_sample)
print(f"Hasil Akhir K-Means (K=4) | Davies-Bouldin: {db_kmeans:.4f} | Silhouette: {sil_kmeans:.4f}")

# Loop untuk mencari nilai K lain (untuk grafik Elbow Curve)
print("\nMelatih ulang (K=2 sd 8) untuk data grafik Elbow & Silhouette K-Means...")
k_range = range(2, 9)
inertia_list = []
silhouette_list = []

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels_temp = kmeans_temp.fit_predict(X_sample)
    
    inertia_list.append(kmeans_temp.inertia_)
    silhouette_list.append(silhouette_score(X_sample, labels_temp))
    print(f"   -> Selesai menghitung untuk K={k}")

# Ekspor Data K-Means
df_metrics_kmeans = pd.DataFrame({'K': k_range, 'Inertia': inertia_list, 'Silhouette_Score': silhouette_list})
df_metrics_kmeans.to_csv("evaluasi_kmeans_elbow_sil.csv", index=False)
kmeans_profile.to_csv("profil_kmeans_heatmap_radar.csv")

In [ ]:
#Klustering Gaussian MIxture Model

In [ ]:
print("Memulai proses training Gaussian Mixture Model (GMM)...")

# covariance_type='full' memungkinkan bentuk klaster elips yang fleksibel
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)

# Melatih model dan langsung memprediksi klaster
df_user['cluster_gmm'] = gmm.fit_predict(X_scaled)

print("GMM telah selesai memetakan 5,27 juta pelanggan.")

In [ ]:
print("Membongkar Karakteristik Rata-rata dari Setiap Klaster GMM...")

# Menghitung rata-rata fitur per klaster versi GMM
gmm_profile = df_user.groupby('cluster_gmm')[features].mean().round(2)

# Menghitung populasi di masing-masing klaster GMM
gmm_profile['jumlah_orang'] = df_user.groupby('cluster_gmm')['user_id'].count()

# Mengurutkan dari klaster yang paling ramai
gmm_profile = gmm_profile.sort_values(by='jumlah_orang', ascending=False)

# Menampilkan hasil
gmm_profile

In [ ]:
print("=== 3. EVALUASI GMM ===")

# Menghitung skor untuk K=4 (Model Final kita)
db_gmm = davies_bouldin_score(x_gmm_sample, y_gmm_sample)
sil_gmm = silhouette_score(X_sample, y_gmm_sample)
print(f"Hasil Akhir GMM (K=4) | Davies-Bouldin: {db_gmm:.4f} | Silhouette: {sil_gmm:.4f}")

# Loop untuk mencari nilai K lain (untuk grafik BIC/Elbow versi GMM)
print("\nMelatih ulang (K=2 sd 8) untuk data grafik BIC & Silhouette GMM...")
bic_list = []
sil_gmm_list = []

for k in k_range:
    gmm_temp = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    labels_temp = gmm_temp.fit_predict(X_sample)
    
    bic_list.append(gmm_temp.bic(X_sample)) # BIC adalah pengganti Inertia pada GMM
    sil_gmm_list.append(silhouette_score(X_sample, labels_temp))
    print(f"   -> Selesai menghitung untuk K={k}")

# Ekspor Data GMM
df_metrics_gmm = pd.DataFrame({'K': k_range, 'BIC_Score': bic_list, 'Silhouette_Score': sil_gmm_list})
df_metrics_gmm.to_csv("evaluasi_gmm_bic_sil.csv", index=False)
gmm_profile.to_csv("profil_gmm_heatmap_radar.csv")

In [ ]:
print("=== 4. REDUKSI DIMENSI (PCA) UNTUK SCATTER PLOT 2D ===")

# Mengompres data dari 6 dimensi menjadi 2 dimensi (X dan Y)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sample)

# Memasukkan hasil koordinat beserta label klaster dari kedua model
df_pca_vis = pd.DataFrame({
    'pca_1': X_pca[:, 0], # Koordinat Sumbu X
    'pca_2': X_pca[:, 1], # Koordinat Sumbu Y
    'cluster_kmeans': y_kmeans_sample,
    'cluster_gmm': y_gmm_sample
})

# Ekspor Data PCA
df_pca_vis.to_csv("data_pca_scatter.csv", index=False)

In [ ]:

comparison = pd.DataFrame({
    'Metrik'            : ['Silhouette Score', 'Davies-Bouldin Score'],
    'K-Means'           : [sil_kmeans, db_kmeans],
    'GMM'               : [sil_gmm, db_gmm]
})

print(comparison.to_string(index=False))
comparison.to_csv('perbandingan_kmeans_vs_gmm.csv', index=False)
print("\nperbandingan_kmeans_vs_gmm.csv siap")